In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


### Creating dataFrames, merging with LEFT JOIN

In [2]:
df_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
df_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

df = df_transaction.merge(df_identity, on='TransactionID', how='left')


del df_transaction, df_identity

In [3]:
df.shape

(590540, 434)

In [ ]:
df.head()

### Splitting data into train and test for later validation

In [4]:
from sklearn.model_selection import train_test_split

features = df.drop(columns=['isFraud'])
target = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

### Exploratory Data Analysis

#### Generally looking at data

In [ ]:
X_train.shape, y_train.shape

In [ ]:
X_train.describe()

#### Checking for NULL entries

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

null_rates = X_train.isna().mean().sort_values(ascending=False)
null_rates = null_rates[null_rates > 0]

plt.figure(figsize=(18, 5))
sns.barplot(x=null_rates.index, y=null_rates.values, palette='magma')
plt.xticks(rotation=90, fontsize=6)
plt.axhline(0.8, color='red', linestyle='--', label='80% threshold')
plt.axhline(0.5, color='orange', linestyle='--', label='50% threshold')
plt.title('Null Rate per Column (train set)', fontsize=14)
plt.ylabel('Null Rate')
plt.xlabel('Column')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Columns with >50% nulls : {(null_rates > 0.5).sum()}")
print(f"Columns with >80% nulls : {(null_rates > 0.8).sum()}")

#### Looking at similar columns

In [ ]:
# id columns 
id_cols = [col for col in X_train.columns if "id" in col]
df[id_cols].head()

In [ ]:
# card columns
card_cols = [col for col in X_train.columns if "card" in col]
df[card_cols].head()

In [ ]:
# C columns 
C_cols = [col for col in X_train.columns if "C" in col]
df[C_cols].head()

In [ ]:
# D columns 
D_cols = [col for col in X_train.columns if "D" in col]
df[D_cols].head()

In [ ]:
# M columns 
M_cols = [col for col in X_train.columns if "M" in col]

df[M_cols].head()

In [ ]:
# V columns 
V_cols = [col for col in X_train.columns if "V" in col]
df[V_cols].head()

#### Categorical and Numerical columns

In [ ]:
numerical_part_df = df.select_dtypes(include=['int64', 'float64'])
categorical_part_df = df.select_dtypes(exclude=['int64', 'float64'])

##### Analyzing Numerical columns 

In [ ]:
numerical_part_df.describe()

##### Analyzing Categorical columns

In [ ]:
categorical_part_df.describe()

#### Looking at target

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

counts = y_train.value_counts()
percentages = y_train.value_counts(normalize=True) * 100

plt.figure(figsize=(8, 6))
ax = sns.barplot(x=counts.index, y=counts.values, palette='coolwarm')

for i, p in enumerate(ax.patches):
    label = f'{counts.iloc[i]}\n({percentages.iloc[i]:.2f}%)'
    ax.annotate(label, 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 15), 
                textcoords='offset points',
                fontsize=11, fontweight='bold')

plt.title('Fraud Class Distribution', fontsize=14)
plt.xlabel('isFraud (0 = Legit, 1 = Fraud)', fontsize=12)
plt.ylabel('Transaction Count', fontsize=12)
plt.ylim(0, counts.max() * 1.2) 
plt.show()

### Preprocessing

#### Writing Preprocessor class

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

class Preprocessor(BaseEstimator, TransformerMixin) : 
    
    def __init__(self, null_tolerance=0.8, inbetween_corr_treshold=0.95,target_corr_treshold=0.01) :
        self.null_tolerance = null_tolerance 
        self.target_corr_treshold = target_corr_treshold 
        self.inbetween_corr_treshold = inbetween_corr_treshold
        self.woe_encoder_ = None
        self.OH_encoder_ = None

    
    def fit(self, X_train, y_train) :
        # columns needed for feature engineering
        self.v_cols = [col for col in X_train.columns if col.startswith('V')]

        self.engineered_features = ['total_V', 'avg_V', 'max_V', 'min_V']
        
        X_temp = self.create_new_features(X_train)
        
        self.categorical_df = X_temp.select_dtypes(exclude=['int64', 'float64'])
        null_series = X_temp.isna().mean()

        # columns that essentially have no substantial data,because more than 80% of the data is null 
        self.garbage_cols = null_series[null_series > self.null_tolerance].index.tolist()

        # columns that store no patterns like identification, or columns with all same elements 
        self.cols_with_no_patterns = self._identify_non_informative_cols(X_temp)

        # columns with numerical data
        self.numerical_cols = X_temp.select_dtypes(include=['int64', 'float64']).columns

        # columns with categorical data 
        self.categorical_cols = self.categorical_df.columns

        # columns for one hot encoding
        self.ohe_cols = self.categorical_df[[col for col in self.categorical_df.columns if self.categorical_df.nunique()[col] <=5]].columns

        # columns with high correlation inbetween each other
        self.high_corr_inbetween = self._identify_high_corr_inbetween(X_temp)

        # columns with low correlation with respect to target
        self.low_corr_with_target = self._identify_low_corr_with_target(X_temp)

        # initialize OH_encoder_
        from category_encoders import OneHotEncoder
        self.OH_encoder_ = OneHotEncoder(cols=self.ohe_cols, use_cat_names=True)
        self.OH_encoder_.fit(X_temp)

        # initialize woe_encoder_ 
        from category_encoders import WOEEncoder
        self.woe_encoder_ = WOEEncoder(cols=[col for col in self.categorical_cols if col not in self.ohe_cols])
        self.woe_encoder_.fit(X_temp, y_train)
        
        return self 
        
    def transform(self, X):  
        X_out = self.create_new_features(X.copy())

        to_drop = list(set(
            self.garbage_cols + 
            self.cols_with_no_patterns + 
            self.high_corr_inbetween + 
            self.low_corr_with_target
        ))

        if self.woe_encoder_:
            X_out = self.woe_encoder_.transform(X_out)
        
        if self.OH_encoder_: 
            X_out = self.OH_encoder_.transform(X_out)

        
        X_out = X_out.drop(columns=[col for col in to_drop if col not in self.engineered_features], errors='ignore')
        
        X_out = X_out.fillna(-999)
    
        return X_out


    def create_new_features(self, X_train) :
        X_out = X_train.copy()
        
        # find v_columns and create new columns from them, such as average of each row 
        X_out['total_V'] = X_out[self.v_cols].sum(axis=1)
        X_out['avg_V'] = X_out[self.v_cols].mean(axis=1)
        X_out['max_V'] = X_out[self.v_cols].max(axis=1)
        X_out['min_V'] = X_out[self.v_cols].min(axis=1)

        return X_out
        

    def _identify_non_informative_cols(self, X_train) -> list[str] : 
        non_informative_cols = [] 
        row_count = len(X_train)
        
        for col in X_train.columns : 
            unique_elem_count = X_train[col].nunique()

            # if all data is same, then column is useless 
            if unique_elem_count <= 1 : 
                non_informative_cols.append(col) 

            # if most of the entries are different, there is no use for that column 
            if unique_elem_count >= (0.99*row_count) : 
                non_informative_cols.append(col)
        
        return non_informative_cols  


    def _identify_high_corr_inbetween(self, X_train) -> list[str] :
        X_temp = self._clone_X_train(X_train)
        
        corr_matrix = X_temp.corr().abs()
        
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        return [column for column in upper.columns if any(upper[column] > self.inbetween_corr_treshold)]
        

    def _identify_low_corr_with_target(self, X_train) -> list[str] :
        X_temp = self._clone_X_train(X_train)
        
        corr_with_target = X_temp.corrwith(y_train).abs().sort_values(ascending=False)
        
        return [col for col in X_temp.columns if corr_with_target[col] <= self.target_corr_treshold]



    def _clone_X_train(self, X_train) :
        X_temp = X_train.copy()
        for col in self.categorical_cols:
            X_temp[col] = X_temp[col].astype('category').cat.codes
        
        X_temp = X_temp.fillna(-999)
        
        return X_temp

### MLFlow and Dagshub setup

In [8]:


import mlflow.sklearn
import dagshub

mlflow.set_experiment("DecisionTree_Training")
mlflow.set_tracking_uri("https://dagshub.com/gbera23-dev/Machine-Learning.mlflow")

dagshub.init(repo_owner='gbera23-dev', repo_name='Machine-Learning', mlflow=True)

2026/05/03 07:35:56 INFO mlflow.tracking.fluent: Experiment with name 'DecisionTree_Training' does not exist. Creating a new experiment.


Initialized MLflow to track repo "gbera23-dev/Machine-Learning"

Repository gbera23-dev/Machine-Learning initialized!

### Selecting first 10 000 samples

In [9]:
SAMPLE_SIZE = 10_000

X_train_sample = X_train.iloc[:SAMPLE_SIZE].reset_index(drop=True)
y_train_sample = y_train.iloc[:SAMPLE_SIZE].reset_index(drop=True)
X_test_sample  = X_test.iloc[:SAMPLE_SIZE].reset_index(drop=True)
y_test_sample  = y_test.iloc[:SAMPLE_SIZE].reset_index(drop=True)

print(f"Train sample : {X_train_sample.shape}")
print(f"Test  sample : {X_test_sample.shape}")

Train sample : (10000, 433)
Test  sample : (10000, 433)


### Instantiating Preprocessor and fitting on X_train_sample

In [ ]:
preprocessor = Preprocessor()
preprocessor.fit(X_train_sample, y_train_sample)
X_out = preprocessor.transform(X_train_sample)

### MLflow logging — preprocessing runs

In [ ]:
# Cleaning run
with mlflow.start_run(run_name="DecisionTree_Cleaning"):
    mlflow.log_param("null_tolerance", preprocessor.null_tolerance)
    mlflow.log_param("garbage_cols_count", len(preprocessor.garbage_cols))
    mlflow.log_param("non_informative_cols_count", len(preprocessor.cols_with_no_patterns))
    mlflow.log_param("fillna_strategy", "fill_with_-999")
    print(f"Garbage cols dropped  : {len(preprocessor.garbage_cols)}")
    print(f"Non-informative cols  : {len(preprocessor.cols_with_no_patterns)}")

# Engineering run
with mlflow.start_run(run_name="DecisionTree_Feature_Engineering"):
    mlflow.log_param("engineered_features", str(preprocessor.engineered_features))
    mlflow.log_param("v_cols_count", len(preprocessor.v_cols))
    mlflow.log_param("woe_encoded_cols_count",
                     len([c for c in preprocessor.categorical_cols if c not in preprocessor.ohe_cols]))
    mlflow.log_param("ohe_encoded_cols_count", len(preprocessor.ohe_cols))
    print(f"New features created  : {preprocessor.engineered_features}")

# Selection run
with mlflow.start_run(run_name="DecisionTree_Feature_Selection"):
    mlflow.log_param("inbetween_corr_threshold", preprocessor.inbetween_corr_treshold)
    mlflow.log_param("target_corr_threshold", preprocessor.target_corr_treshold)
    mlflow.log_param("high_corr_dropped_count", len(preprocessor.high_corr_inbetween))
    mlflow.log_param("low_target_corr_dropped_count", len(preprocessor.low_corr_with_target))
    mlflow.log_param("remaining_features", len(X_out.columns))
    print(f"High inter-corr cols dropped : {len(preprocessor.high_corr_inbetween)}")
    print(f"Low target-corr cols dropped : {len(preprocessor.low_corr_with_target)}")
    print(f"Features remaining           : {len(X_out.columns)}")

### Grid Search over Decision Tree hyperparameters

#### Each combination is trained on the 10 000-row sample, depth is not restricted

In [19]:
import itertools
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score


param_grid = {
    "criterion"        : ["gini"],
    "max_depth"        : [None],
    "min_samples_split": [10, 20],
    "min_samples_leaf" : [3, 5, 20],
}

keys   = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f"Total grid-search runs : {len(combos)}")

N_SPLITS    = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

best_val_auc  = -1
best_params   = None
results_log   = []          


for run_idx, combo in enumerate(combos, start=1):
    params = dict(zip(keys, combo))

    train_auc_scores = []
    val_auc_scores   = []

    X_cv = X_train_sample.reset_index(drop=True)
    y_cv = y_train_sample.reset_index(drop=True)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_cv, y_cv)):
        X_tr, y_tr = X_cv.iloc[train_idx], y_cv.iloc[train_idx]
        X_va, y_va = X_cv.iloc[val_idx],   y_cv.iloc[val_idx]

        fold_preprocessor = Preprocessor(
            null_tolerance=0.8,
            inbetween_corr_treshold=0.95,
            target_corr_treshold=0.01
        )
        X_tr_proc = fold_preprocessor.fit(X_tr, y_tr).transform(X_tr)
        X_va_proc = fold_preprocessor.transform(X_va)

        fold_model = DecisionTreeClassifier(
            criterion=params["criterion"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=RANDOM_STATE
        )
        fold_model.fit(X_tr_proc, y_tr)

        train_auc_scores.append(roc_auc_score(y_tr, fold_model.predict_proba(X_tr_proc)[:, 1]))
        val_auc_scores.append(roc_auc_score(y_va,   fold_model.predict_proba(X_va_proc)[:, 1]))

    mean_train_auc = float(np.mean(train_auc_scores))
    mean_val_auc   = float(np.mean(val_auc_scores))
    std_val_auc    = float(np.std(val_auc_scores))
    auc_gap        = mean_train_auc - mean_val_auc

    if mean_val_auc > best_val_auc:
        best_val_auc = mean_val_auc
        best_params  = params.copy()


    run_name = (
        f"DT_grid_{run_idx:03d}_"
        f"{params['criterion']}_"
        f"depth{'unlimited depth'}_"
        f"split{params['min_samples_split']}_"
        f"leaf{params['min_samples_leaf']}"
    )

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("train_size",         SAMPLE_SIZE)
        mlflow.log_param("criterion",          params["criterion"])
        mlflow.log_param("max_depth",          str(params["max_depth"]))
        mlflow.log_param("min_samples_split",  params["min_samples_split"])
        mlflow.log_param("min_samples_leaf",   params["min_samples_leaf"])
        mlflow.log_param("cv_folds",           N_SPLITS)

        mlflow.log_metric("mean_train_auc", mean_train_auc)
        mlflow.log_metric("mean_val_auc",   mean_val_auc)
        mlflow.log_metric("std_val_auc",    std_val_auc)
        mlflow.log_metric("auc_gap",        auc_gap)

    results_log.append({
        "run_idx"          : run_idx,
        **params,
        "mean_train_auc"   : mean_train_auc,
        "mean_val_auc"     : mean_val_auc,
        "std_val_auc"      : std_val_auc,
        "auc_gap"          : auc_gap,
    })

    print(
        f"[{run_idx:>3}/{len(combos)}] {params}  "
        f"Val AUC: {mean_val_auc:.4f} ± {std_val_auc:.4f}  "
        f"Gap: {auc_gap:.4f}"
    )

print("\n" + "=" * 60)
print(f"Best Val AUC  : {best_val_auc:.4f}")
print(f"Best params   : {best_params}")

Total grid-search runs : 6


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:292

🏃 View run DT_grid_001_gini_depthunlimited depth_split10_leaf3 at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/558a0196ef8b40129471e253c68748bd
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
[  1/6] {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 3}  Val AUC: 0.6570 ± 0.0145  Gap: 0.3320


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:292

🏃 View run DT_grid_002_gini_depthunlimited depth_split10_leaf5 at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/5978941c692c4d1a8cc80192b15f1469
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
[  2/6] {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 5}  Val AUC: 0.6632 ± 0.0203  Gap: 0.3223


...

🏃 View run DT_grid_004_gini_depthunlimited depth_split20_leaf3 at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/a8ee1c3baaeb41b79223c7620c82a3ca
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
[  4/6] {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 20, 'min_samples_leaf': 3}  Val AUC: 0.6742 ± 0.0231  Gap: 0.3061


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:292

🏃 View run DT_grid_005_gini_depthunlimited depth_split20_leaf5 at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/2deae032e69141bf9c58db0508e0e4af
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
[  5/6] {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 20, 'min_samples_leaf': 5}  Val AUC: 0.6857 ± 0.0242  Gap: 0.2916


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:292

🏃 View run DT_grid_006_gini_depthunlimited depth_split20_leaf20 at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/df00864c94744ac9a1cf9413400b1a8c
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
[  6/6] {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 20, 'min_samples_leaf': 20}  Val AUC: 0.7384 ± 0.0353  Gap: 0.2132

Best Val AUC  : 0.7384
Best params   : {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 20}


### Grid search Summary for unlimited depth decision trees

In [20]:
results_df = pd.DataFrame(results_log).sort_values("mean_val_auc", ascending=False)
results_df.head(10)

,run_idx,criterion,max_depth,min_samples_split,min_samples_leaf,mean_train_auc,mean_val_auc,std_val_auc,auc_gap
5,6,gini,None,20,20,0.951561,0.738370,0.035271,0.213191
2,3,gini,None,10,20,0.951561,0.738370,0.035271,0.213191
4,5,gini,None,20,5,0.977291,0.685697,0.024169,0.291594
3,4,gini,None,20,3,0.980389,0.674249,0.023071,0.306140
1,2,gini,None,10,5,0.985428,0.663163,0.020317,0.322264
0,1,gini,None,10,3,0.988929,0.656957,0.014508,0.331972


#### Each combination is trained on the 10 000-row sample, evaluated with 5-fold stratified CV, and logged as its own MLflow run so results are comparable side-by-side.

In [12]:
param_grid = {
    "criterion"        : ["gini"],
    "max_depth"        : [4, 7, 10],
    "min_samples_split": [10, 20],
    "min_samples_leaf" : [3, 5, 20],
}

keys   = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f"Total grid-search runs : {len(combos)}")

N_SPLITS    = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

best_val_auc  = -1
best_params   = None
results_log   = []          


for run_idx, combo in enumerate(combos, start=1):
    params = dict(zip(keys, combo))

    train_auc_scores = []
    val_auc_scores   = []

    X_cv = X_train_sample.reset_index(drop=True)
    y_cv = y_train_sample.reset_index(drop=True)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_cv, y_cv)):
        X_tr, y_tr = X_cv.iloc[train_idx], y_cv.iloc[train_idx]
        X_va, y_va = X_cv.iloc[val_idx],   y_cv.iloc[val_idx]

        fold_preprocessor = Preprocessor(
            null_tolerance=0.8,
            inbetween_corr_treshold=0.95,
            target_corr_treshold=0.01
        )
        X_tr_proc = fold_preprocessor.fit(X_tr, y_tr).transform(X_tr)
        X_va_proc = fold_preprocessor.transform(X_va)

        fold_model = DecisionTreeClassifier(
            criterion=params["criterion"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=RANDOM_STATE
        )
        fold_model.fit(X_tr_proc, y_tr)

        train_auc_scores.append(roc_auc_score(y_tr, fold_model.predict_proba(X_tr_proc)[:, 1]))
        val_auc_scores.append(roc_auc_score(y_va,   fold_model.predict_proba(X_va_proc)[:, 1]))

    mean_train_auc = float(np.mean(train_auc_scores))
    mean_val_auc   = float(np.mean(val_auc_scores))
    std_val_auc    = float(np.std(val_auc_scores))
    auc_gap        = mean_train_auc - mean_val_auc

    if mean_val_auc > best_val_auc:
        best_val_auc = mean_val_auc
        best_params  = params.copy()


    run_name = (
        f"DT_grid_{run_idx:03d}_"
        f"{params['criterion']}_"
        f"depth{params['max_depth']}_"
        f"split{params['min_samples_split']}_"
        f"leaf{params['min_samples_leaf']}"
    )

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("train_size",         SAMPLE_SIZE)
        mlflow.log_param("criterion",          params["criterion"])
        mlflow.log_param("max_depth",          str(params["max_depth"]))
        mlflow.log_param("min_samples_split",  params["min_samples_split"])
        mlflow.log_param("min_samples_leaf",   params["min_samples_leaf"])
        mlflow.log_param("cv_folds",           N_SPLITS)

        mlflow.log_metric("mean_train_auc", mean_train_auc)
        mlflow.log_metric("mean_val_auc",   mean_val_auc)
        mlflow.log_metric("std_val_auc",    std_val_auc)
        mlflow.log_metric("auc_gap",        auc_gap)

    results_log.append({
        "run_idx"          : run_idx,
        **params,
        "mean_train_auc"   : mean_train_auc,
        "mean_val_auc"     : mean_val_auc,
        "std_val_auc"      : std_val_auc,
        "auc_gap"          : auc_gap,
    })

    print(
        f"[{run_idx:>3}/{len(combos)}] {params}  "
        f"Val AUC: {mean_val_auc:.4f} ± {std_val_auc:.4f}  "
        f"Gap: {auc_gap:.4f}"
    )

print("\n" + "=" * 60)
print(f"Best Val AUC  : {best_val_auc:.4f}")
print(f"Best params   : {best_params}")

Total grid-search runs : 18


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:292

🏃 View run DT_grid_001_gini_depth4_split10_leaf3 at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/c4ecd47f5b254923b75c0f8f72fb30d3
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
[  1/18] {'criterion': 'gini', 'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 3}  Val AUC: 0.7436 ± 0.0397  Gap: 0.0308


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:292

🏃 View run DT_grid_002_gini_depth4_split10_leaf5 at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/73529a49deb34dbc970fac9a7d0c76e5
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
[  2/18] {'criterion': 'gini', 'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 5}  Val AUC: 0.7528 ± 0.0307  Gap: 0.0336


...

🏃 View run DT_grid_016_gini_depth10_split20_leaf3 at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/ab8336d77dde4a48a054de77f7cefdd9
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
[ 16/18] {'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 20, 'min_samples_leaf': 3}  Val AUC: 0.6718 ± 0.0188  Gap: 0.2339


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:292

🏃 View run DT_grid_017_gini_depth10_split20_leaf5 at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/3d749609eed04dc4ae73129ab68e5c8f
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
[ 17/18] {'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 20, 'min_samples_leaf': 5}  Val AUC: 0.6749 ± 0.0485  Gap: 0.2415


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:292

🏃 View run DT_grid_018_gini_depth10_split20_leaf20 at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/257951862abc4eedb786c4b90ad28e2c
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
[ 18/18] {'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 20, 'min_samples_leaf': 20}  Val AUC: 0.7230 ± 0.0574  Gap: 0.1965

Best Val AUC  : 0.7626
Best params   : {'criterion': 'gini', 'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 20}


### Grid search results summary

In [13]:
results_df = pd.DataFrame(results_log).sort_values("mean_val_auc", ascending=False)
results_df.head(10)

,run_idx,criterion,max_depth,min_samples_split,min_samples_leaf,mean_train_auc,mean_val_auc,std_val_auc,auc_gap
5,6,gini,4,20,20,0.792594,0.762567,0.025687,0.030027
2,3,gini,4,10,20,0.792594,0.762567,0.025687,0.030027
1,2,gini,4,10,5,0.786473,0.752826,0.030676,0.033646
4,5,gini,4,20,5,0.785962,0.752400,0.030640,0.033562
0,1,gini,4,10,3,0.774409,0.743617,0.039704,0.030791
3,4,gini,4,20,3,0.773812,0.743243,0.039439,0.030569
8,9,gini,7,10,20,0.871848,0.737302,0.048374,0.134546
11,12,gini,7,20,20,0.871848,0.737302,0.048374,0.134546
9,10,gini,7,20,3,0.854856,0.737131,0.037583,0.117725
10,11,gini,7,20,5,0.862855,0.732964,0.036939,0.129892


### Training final model with best hyperparameters

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

preprocessor_final = Preprocessor(
    null_tolerance=0.8,
    inbetween_corr_treshold=0.95,
    target_corr_treshold=0.01
)

X_train_processed = preprocessor_final.fit(X_train_sample, y_train_sample).transform(X_train_sample)
X_test_processed  = preprocessor_final.transform(X_test_sample)

model = DecisionTreeClassifier(
    criterion=best_params["criterion"],
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    min_samples_leaf=best_params["min_samples_leaf"],
    random_state=RANDOM_STATE
)

model.fit(X_train_processed, y_train_sample)
print("Final model trained with best params:", best_params)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Final model trained with best params: {'criterion': 'gini', 'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 20}


### Evaluating and logging the final model

In [17]:
test_preds_proba = model.predict_proba(X_test_processed)[:, 1]
test_preds       = model.predict(X_test_processed)

test_auc      = roc_auc_score(y_test_sample, test_preds_proba)
test_accuracy = accuracy_score(y_test_sample, test_preds)

print(f"Test AUC      : {test_auc:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_sample, test_preds))

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_final),
    ('model',        model)
])

with mlflow.start_run(run_name="DecisionTree_SmallData_BestModel_Evaluation"):

    mlflow.log_param("train_size",          SAMPLE_SIZE)
    mlflow.log_param("criterion",           best_params["criterion"])
    mlflow.log_param("max_depth",           str(best_params["max_depth"]))
    mlflow.log_param("min_samples_split",   best_params["min_samples_split"])
    mlflow.log_param("min_samples_leaf",    best_params["min_samples_leaf"])

    mlflow.log_metric("best_mean_cv_val_auc", best_val_auc)

    mlflow.log_metric("test_auc",      test_auc)
    mlflow.log_metric("test_accuracy", test_accuracy)

    # mlflow.sklearn.log_model(final_pipeline, artifact_path="decision_tree_pipeline")

Test AUC      : 0.7624
Test Accuracy : 0.9653

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      9649
           1       0.52      0.12      0.20       351

    accuracy                           0.97     10000
   macro avg       0.75      0.56      0.59     10000
weighted avg       0.95      0.97      0.95     10000

🏃 View run DecisionTree_SmallData_BestModel_Evaluation at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4/runs/423954db995e44e398402c2697539599
🧪 View experiment at: https://dagshub.com/gbera23-dev/Machine-Learning.mlflow/#/experiments/4
